# Customer Churn Prediction

## Data Preprocessing

### Objective

The purpose of this notebook is to prepare the dataset for machine learning by cleaning the data, encoding categorical variables, scaling numerical features, and splitting the dataset into training and testing subsets.

In [69]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [70]:
#Load dataset
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df_copy = df.copy()

In [71]:
df_copy["TotalCharges"] = pd.to_numeric(
    df_copy["TotalCharges"],
    errors="coerce"
)
df_copy["TotalCharges"].dtype
df_copy["TotalCharges"].isnull().sum()

np.int64(11)

### Observation

Converting TotalCharges to a numeric format revealed a small number of missing values caused by blank entries.

# Missing Value Treatment

In [72]:
df_copy[df_copy["TotalCharges"].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [73]:
df_copy["TotalCharges"].fillna(df_copy["TotalCharges"].median(), inplace=True)


C:\Users\molka\AppData\Local\Temp\ipykernel_14008\358192057.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy["TotalCharges"].fillna(df_copy["TotalCharges"].median(), inplace=True)


In [74]:
df_copy["TotalCharges"].isnull().sum()

np.int64(0)

### Observation

Missing values were replaced using the median, which is robust to extreme values.

# Feature Selection

In [75]:
df_copy.drop("customerID", axis=1, inplace=True)
df_copy.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Observation

The customer identifier was removed because it does not contain predictive information and is unique for each customer.

# Target Variable Encoding

In [76]:
df_copy["churn"] = df_copy["Churn"].map({"Yes": 1, "No": 0})
df_copy["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

### Observation

The target variable was converted into a binary numerical format suitable for machine learning algorithms.

# Feature Encoding

Machine learning algorithms require numerical inputs.

Categorical variables are therefore converted into numerical representations.

In [77]:
categorical_cols = df_copy.select_dtypes(
    include="object"
).columns

categorical_cols

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='object')

In [78]:
df_encoded = pd.get_dummies(
    df_copy,
    columns=categorical_cols,
    drop_first=True
)
df_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,True,False,True,False,True


In [79]:
df_encoded.shape

(7043, 32)

### Observation

Categorical features were transformed using One-Hot Encoding, allowing machine learning algorithms to process them effectively.

# Feature-Target Separation

In [80]:
X = df_encoded.drop("churn", axis=1)

y = df_encoded["churn"]
print(X.shape)
print(y.shape)

(7043, 31)
(7043,)


# Train-Test Split

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [82]:
print(X_train.shape)
print(X_test.shape)

(5634, 31)
(1409, 31)


### Observation

The dataset was divided into training and testing subsets.

The training set will be used for model learning, while the testing set will be reserved for performance evaluation.

In [83]:
print("Train Churn Rate:")
print(y_train.mean())

print("\nTest Churn Rate:")
print(y_test.mean())

Train Churn Rate:
0.2653532126375577

Test Churn Rate:
0.2654364797728886


### Observation

The churn proportions in the training and testing datasets remain similar due to stratified sampling.

This ensures that both subsets are representative of the original dataset.

# Feature Scaling

In [84]:
numerical_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
scaler = StandardScaler()

X_train[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_test[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)
X_train.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
3738,0,0.102371,-0.521976,-0.263289,True,False,False,False,True,False,...,True,False,True,False,False,False,False,True,False,False
3151,0,-0.711743,0.337478,-0.504814,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,True,False
4860,0,-0.793155,-0.809013,-0.751213,True,True,True,False,True,False,...,False,False,False,False,True,False,False,False,True,False
3867,0,-0.263980,0.284384,-0.173699,False,True,False,True,False,False,...,True,False,True,False,True,True,True,False,False,False
3810,0,-1.281624,-0.676279,-0.990851,True,True,True,True,False,False,...,False,False,False,False,False,False,False,True,False,False


### Observation

Numerical features were standardized to ensure that variables with larger scales do not dominate the learning process.

# Save Processed Data

In [85]:
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

### Observation

The processed datasets were saved for reuse in the modeling phase.

# Conclusion

The dataset has been successfully prepared for machine learning.

The preprocessing steps included:

- Data type correction
- Missing value treatment
- Feature selection
- Target encoding
- One-hot encoding
- Train-test splitting
- Feature scaling

The resulting datasets are now ready for model development and evaluation.